In [1]:
# Все библиотеки

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import ttest_ind, levene, mannwhitneyu,shapiro, \
    f_oneway, kruskal, chi2_contingency, fisher_exact
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import pingouin as pg
import sys
import os
from ydata_profiling import ProfileReport
from matplotlib import rcParams
from matplotlib import font_manager as fm
import matplotlib as mpl
import warnings

mpl.rcParams['font.family'] = 'HSE Sans'

warnings.filterwarnings('ignore')

# Графики из graphics.ipynb

In [ ]:
def plot_bar_chart(x, y, title=None, color='#4A90E2', bgcolor='#F5F5F5', figsize=(8, 5)):
    """
    Рисует столбчатую диаграмму с подписями значений над столбцами.

    x : list/array-like — подписи для оси X
    y : list/array-like — высоты столбцов
    title : str — заголовок графика
    color : str — цвет столбцов (hex или имя)
    bgcolor : str — цвет фона графика
    figsize : pair - figsize
    """
    fig, ax = plt.subplots(figsize=figsize)

    # Фон
    fig.patch.set_facecolor(bgcolor)
    ax.set_facecolor(bgcolor)

    # Построение столбцов
    bars = ax.bar(x, y, color=color, edgecolor='black')

    # Подписи над столбцами
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # отступ в пикселях
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=10, color='black')

    # Заголовок
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')

    # Убираем рамку сверху и справа (минимализм)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()


In [ ]:
from matplotlib.colors import Normalize, TwoSlopeNorm


def plot_heatmap(data,
                 x_labels=None,
                 y_labels=None,
                 title=None,
                 cmap='Blues',
                 bgcolor='#F5F5F5',
                 figsize=(8, 6),
                 annot=True,
                 fmt='.2f',
                 cbar_label=None,
                 vmin=None,
                 vmax=None,
                 center=None,
                 linewidths=0.5,
                 linecolor='white'):
    """
    Рисует аккуратную heatmap.

    Параметры (кратко):
    - data : 2D array-like (список списков / numpy array)
    - x_labels, y_labels : подписи для колонок/строк (по умолчанию индексы)
    - title : заголовок графика
    - cmap : цветовая карта (по умолчанию 'Blues' — в твоём синем стиле)
    - bgcolor : цвет фона фигуры
    - figsize : размер фигуры
    - annot : подписывать значения в ячейках
    - fmt : формат подписей (строка форматирования, например '.2f' или 'd')
    - cbar_label : подпись для colorbar
    - vmin, vmax : ручной минимум/максимум для cmap. По умолчанию берутся из данных.
    - center : центр для двухцветной нормировки (если нужно центрировать ноль) — тогда используется TwoSlopeNorm
    - linewidths, linecolor : тонкие линии между ячейками

    Функция корректно обрабатывает NaN (оставляет пустые ячейки) и автоматически выбирает цвет текста
    (чёрный для светлых ячеек, белый для тёмных) по относительной яркости.
    """

    data = np.array(data, dtype=float)
    if data.ndim != 2:
        raise ValueError("data must be 2D (matrix-like)")

    nrows, ncols = data.shape

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(bgcolor)
    ax.set_facecolor(bgcolor)

    # пределы
    if vmin is None:
        # игнорируем NaN при поиске минимума/максимума
        vmin = np.nanmin(data)
    if vmax is None:
        vmax = np.nanmax(data)

    # нормировка (центрируем, если надо)
    if center is not None:
        norm = TwoSlopeNorm(vcenter=center, vmin=vmin, vmax=vmax)
    else:
        norm = Normalize(vmin=vmin, vmax=vmax)

    im = ax.imshow(data, cmap=cmap, norm=norm, aspect='auto', interpolation='nearest')

    # colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if cbar_label:
        cbar.set_label(cbar_label, fontsize=10)

    # подписи осей
    if x_labels is None:
        x_labels = [str(i) for i in range(ncols)]
    if y_labels is None:
        y_labels = [str(i) for i in range(nrows)]

    ax.set_xticks(np.arange(ncols))
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(nrows))
    ax.set_yticklabels(y_labels, fontsize=9)

    # тонкие линии между ячейками
    ax.set_xticks(np.arange(-0.5, ncols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, nrows, 1), minor=True)
    ax.grid(which='minor', color=linecolor, linestyle='-', linewidth=linewidths)
    ax.tick_params(which='minor', bottom=False, left=False)

    # подписи в ячейках
    if annot:
        # защитимся от деления на ноль при одинаковых vmin/vmax
        denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0
        for i in range(nrows):
            for j in range(ncols):
                val = data[i, j]
                if np.isnan(val):
                    text = ''
                else:
                    try:
                        text = format(val, fmt)
                    except Exception:
                        # если пользователь передал некорректный формат —fallback
                        text = f"{val:.2f}"

                if text != '':
                    # относительная яркость для выбора цвета текста
                    rel = (val - vmin) / denom
                    text_color = 'white' if rel > 0.5 else 'black'
                    ax.text(j, i, text, ha='center', va='center', fontsize=9, color=text_color)

    # заголовок
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')

    # минимализм — убираем рамки сверху и справа
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

# Тесты

## Корреляция (Количественная - Количественная)

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

def print_correlations(x, y):
    # Спирмен
    spearman_corr, spearman_p = spearmanr(x, y, nan_policy='omit')
    # Пирсон
    pearson_corr, pearson_p = pearsonr(x, y)

    print(f"Корреляция Спирмена: {spearman_corr:.2f}, p-value = {spearman_p:.4f}")
    print(f"Корреляция Пирсона: {pearson_corr:.2f}, p-value = {pearson_p:.4f}")

# Пример использования:
# print_correlations(df['x1'], df['x2'])


## n-выборочные тесты (Категориальная - количественная)

In [ ]:
def normal_means(x):
    if x.shape[0] > 30:
        return True
    p_shapiro = shapiro(x)[-1]
    return p_shapiro > 0.05


def equal_var(*s):
    p_levene = levene(*s)[-1]
    if p_levene > 0.05:
        print('Дисперсии выборок равны')
    else:
        print('Дисперсии выборок НЕ равны')
    return p_levene

def compare_mean(df, col1, col2):
    s1 = df[col1]
    s2 = df[col2]
    p_normal_means_s1 = normal_means(s1)
    p_normal_means_s12 = normal_means(s2)
    p_equal_var = equal_var(s1, s2)
    print('p_normal_means_s1:', p_normal_means_s1)
    print('p_normal_means_s12:', p_normal_means_s12)
    print('p_equal_var:', p_equal_var)
    if p_normal_means_s1 > 0.05 and p_normal_means_s12 > 0.05:
        if p_equal_var > 0.05:
            print('Т-Критерий Стьюдента')
            p_test = ttest_ind(s1, s2, equal_var=True)[-1]
        else:
            print('Т-Критерий Уэлча')
            p_test = ttest_ind(s1, s2, equal_var=False)[-1]
    else:
        print('U-критерий Манна-Уитни')
        p_test = mannwhitneyu(s1, s2)[-1]
    print('p-value:', p_test)
    if p_test <= 0.05:
        print('Средние стат. значимо отличаются')
    else:
        print('Средние стат. значимо НЕ отличаются')



def compare_mean(df, num_var, cat_var):
    groups_count = df[cat_var].nunique()
    series_list = [df[df[cat_var] == g][num_var] for g in df[cat_var].unique()]
    test_type = 'parametric'

    # Проверка нормальности
    norm = all(normal_means(s) for s in series_list)

    if not norm:
        if groups_count == 2:
            print("U-критерий Манна-Уитни")
            p = mannwhitneyu(series_list[0], series_list[1])[1]
        else:
            print("Kruskal-Wallis Test")
            test_type = 'non parametric'
            p = kruskal(*series_list)[1]
            if p <= 0.05:
                print("Post-hoc: Dunn test (p-values, Bonferroni corrected):")
                posthoc = sp.posthoc_dunn(df, val_col=num_var, group_col=cat_var, p_adjust='bonferroni')
                print(posthoc)
    else:
        p_equal_var = equal_var(series_list)
        print("p_equal_var:", p_equal_var)
        if p_equal_var > 0.05:
            if groups_count == 2:
                print("Т-критерий Стьюдента")
                p = ttest_ind(series_list[0], series_list[1], equal_var=True)[1]
            else:
                print("One-Way ANOVA")
                test_type = 'non parametric'
                p = f_oneway(*series_list)[1]
                if p <= 0.05:
                    print("Post-hoc: Tukey HSD:")
                    tukey = pairwise_tukeyhsd(endog=df[num_var], groups=df[cat_var], alpha=0.05)
                    print(tukey)
        else:
            if groups_count == 2:
                print("Т-критерий Уэлча")
                p = ttest_ind(series_list[0], series_list[1], equal_var=False)[1]
            else:
                print("Welch's ANOVA")
                test_type = 'non parametric'
                welch = pg.welch_anova(dv=num_var, between=cat_var, data=df)
                p = welch.loc[0, 'p-unc']
                if p <= 0.05:
                    print("Post-hoc: Games-Howell test:")
                    gh = pg.pairwise_gameshowell(dv=num_var, between=cat_var, data=df)
                    print(gh)

    print("p-value:", p)
    if p <= 0.05:
        print("Средние статистически значимо отличаются")
        if test_type == 'parametric':
            agg = 'mean'
        else:
            agg = 'median'
        agg_values = df.groupby(cat_var)[num_var].agg(agg)
        plot_bar_chart(agg_values.index, agg_values.values, title=f"{agg} {num_var} по {cat_var}")
    else:
        print("Средние статистически значимо НЕ отличаются")

## Хи-квадрат (Категориальная - Категориальная)

In [ ]:
def chi_square_test(df, col1, col2, correction=False):
    """
    χ²-тест независимости для двух категориальных переменных.

    df : pd.DataFrame — таблица данных
    col1, col2 : str — названия категориальных столбцов
    correction : bool — Yates' continuity correction (по умолчанию False)

    Выводит:
        - наблюдаемые частоты
        - ожидаемые частоты
        - χ²-статистику, p-value, степени свободы
    """
    # таблица сопряжённости
    contingency_table = pd.crosstab(df[col1], df[col2])

    # χ²-тест
    chi2, p, dof, expected = chi2_contingency(contingency_table, correction=correction)

    print("Наблюдаемые частоты:")
    print(contingency_table, "\n")

    print("Ожидаемые частоты (если бы переменные были независимы):")
    print(pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns), "\n")

    print(f"χ² = {chi2:.4f}, p-value = {p:.4f}, df = {dof}")
    if p <= 0.05:
        print("Связь между переменными статистически значима.")
        plot_heatmap(contingency_table, title="елки иголки")

    else:
        print("Нет статистически значимой связи между переменными.")

    return chi2, p, dof, expected
